Maximizing profitability through prices adjustment 

In [1]:
import gdown
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.optimize import minimize

In [2]:
fct_order  = pd.read_csv('C:/Users/seeno/Documents/Deloitte/erfjiefrjiefjirefjiefji/Menu Engineering Part 2/fct_order_items.csv',low_memory=False)

dim_items = pd.read_csv("C:/Users/seeno/Documents/Deloitte/erfjiefrjiefjirefjiefji/Menu Engineering Part 2/dim_items.csv",low_memory=False)

dim_menu  = pd.read_csv("C:/Users/seeno/Documents/Deloitte/erfjiefrjiefjirefjiefji/Menu Engineering Part 2/dim_menu_items.csv",low_memory=False)

most      = pd.read_csv("C:/Users/seeno/Documents/Deloitte/erfjiefrjiefjirefjiefji/Menu Engineering Part 2/most_ordered.csv",low_memory=False)




In [3]:

Payment_1 = pd.read_csv(
    r"C:/Users/seeno/Documents/Deloitte/erfjiefrjiefjirefjiefji/Menu Engineering Part 2/fct_payments_part1.csv",
    engine="python",
    on_bad_lines="skip"
)

Payment_2 = pd.read_csv(
    r"C:/Users/seeno/Documents/Deloitte/erfjiefrjiefjirefjiefji/Menu Engineering Part 2/fct_payments_part2.csv",
    engine="python",
    on_bad_lines="skip"
)

payments = pd.concat([Payment_1, Payment_2], ignore_index=True)


In [4]:

most_agg = (most.groupby("item_id", as_index=False)
              .agg(order_count=("order_count","sum"),
                   places=("place_id","nunique")))

menu_master = (dim_items
    .merge(dim_menu[["id","section_id","rating","votes","purchases","price","status","type","title","index"]],
           on="id", how="left", suffixes=("","_menu"))
    .merge(most_agg, left_on="id", right_on="item_id", how="left")
)

menu_master.to_csv("menu_master.csv", index=False)


In [5]:
pricing_pool = menu_master[
    (menu_master["status"] == "Active") &
    (menu_master["type"] == "Normal") &
    (menu_master["price"].notna()) &
    (menu_master["price"] > 0)
].copy()


In [6]:
pricing_pool["popularity"] = pricing_pool["order_count"].fillna(0)

q70 = pricing_pool["popularity"].quantile(0.70)
q90 = pricing_pool["popularity"].quantile(0.90)

pricing_pool["role"] = "low_performer"
pricing_pool.loc[pricing_pool["popularity"] >= q70, "role"] = "core"
pricing_pool.loc[pricing_pool["popularity"] >= q90, "role"] = "traffic_driver"


In [7]:
payments.head()

,id,user_id,created,updated,amount,card_number,card_type,cashback,change,channel,...,requires_signature,session_id,status,synchronized_to_accounting,tip,trainee_mode,type,url,voucher_id,seq_number
0,84025,0,1647587029,1656097003,6.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Settled,NaN,0.0,0.0,online,NaN,NaN,1.0
1,84029,0,1647587502,1656097003,36.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Settled,NaN,0.0,0.0,online,NaN,NaN,2.0
2,84040,945,1647592063,1656097003,6.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Settled,NaN,0.0,0.0,online,NaN,NaN,3.0
3,84119,0,1647771527,1656097003,119.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Unsettled,NaN,0.0,0.0,cash,NaN,NaN,NaN
4,84125,0,1647779448,1656097003,119.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Unsettled,NaN,0.0,0.0,cash,NaN,NaN,NaN


In [8]:

fct_order["created_dt"] = pd.to_datetime(fct_order["created"], unit="s", errors="coerce")
fct_order["date"] = fct_order["created_dt"].dt.date


In [9]:
fct_order["gross_revenue"] = fct_order["price"] * fct_order["quantity"]
fct_order["net_revenue"] = fct_order["gross_revenue"] - fct_order["discount_amount"].fillna(0)

# realized unit price (avoid divide by zero)
fct_order["realized_unit_price"] = np.where(
    fct_order["quantity"] > 0,
    fct_order["net_revenue"] / fct_order["quantity"],
    np.nan
)


In [10]:
panel_item_day = (
    fct_order.groupby(["id", "date"], as_index=False)
             .agg(
                 qty=("quantity", "sum"),
                 avg_price=("realized_unit_price", "mean"),
                 revenue=("net_revenue", "sum"),
                 n_orders=("order_id", "nunique")
             )
)

panel_item_day["log_qty"] = np.log(panel_item_day["qty"].clip(lower=1))
panel_item_day["log_price"] = np.log(panel_item_day["avg_price"].clip(lower=0.01))


In [11]:
print([c for c in payments.columns if "order" in c.lower()])
print([c for c in payments.columns if "receipt" in c.lower()])
print([c for c in payments.columns if "sale" in c.lower()])


['order_code']
['receipt_number']
[]


In [12]:
fct_order[["order_id", "id"]].head()


,order_id,id
0,60825.0,60824
1,60841.0,60837
2,60841.0,60838
3,60841.0,60839
4,60841.0,60840


In [13]:
[c for c in fct_order.columns if "order" in c.lower() or "receipt" in c.lower() or "code" in c.lower() or "number" in c.lower()]


['order_id']

In [14]:

# timestamps
fct_order["created_dt"] = pd.to_datetime(
    fct_order["created"], unit="s", errors="coerce"
)
fct_order["date"] = fct_order["created_dt"].dt.date

# revenues
fct_order["gross_revenue"] = fct_order["price"] * fct_order["quantity"]
fct_order["net_revenue"] = (
    fct_order["gross_revenue"] - fct_order["discount_amount"].fillna(0)
)

# realized unit price
fct_order["realized_unit_price"] = np.where(
    fct_order["quantity"] > 0,
    fct_order["net_revenue"] / fct_order["quantity"],
    np.nan
)

# item-day panel (THIS is what elasticity needs)
panel_item_day = (
    fct_order
    .groupby(["id", "date"], as_index=False)
    .agg(
        qty=("quantity", "sum"),
        avg_price=("realized_unit_price", "mean"),
        revenue=("net_revenue", "sum"),
        n_orders=("order_id", "nunique")
    )
)

panel_item_day["log_qty"] = np.log(panel_item_day["qty"].clip(lower=1))
panel_item_day["log_price"] = np.log(panel_item_day["avg_price"].clip(lower=0.01))


In [15]:
panel_item_day = panel_item_day.merge(
    menu_master[["id", "title", "section_id", "price"]],
    on="id",
    how="left"
)


In [16]:
panel_item_day.shape
panel_item_day.head()


,id,date,qty,avg_price,revenue,n_orders,log_qty,log_price,title,section_id,price
0,60824,2021-02-12,2,25.0,50.0,1,0.693147,3.218876,NaN,NaN,NaN
1,60837,2021-02-12,1,175.0,175.0,1,0.000000,5.164786,NaN,NaN,NaN
2,60838,2021-02-12,1,160.0,160.0,1,0.000000,5.075174,NaN,NaN,NaN
3,60839,2021-02-12,1,160.0,160.0,1,0.000000,5.075174,NaN,NaN,NaN
4,60840,2021-02-12,1,160.0,160.0,1,0.000000,5.075174,NaN,NaN,NaN


In [17]:
panel_item_day["id"] = panel_item_day["id"].astype(str)
menu_master["id"] = menu_master["id"].astype(str)


In [18]:
panel_item_day = panel_item_day.merge(
    menu_master[["id", "title", "section_id", "price"]],
    on="id",
    how="left"
)


In [19]:
panel_item_day.head()


,id,date,qty,avg_price,revenue,n_orders,log_qty,log_price,title_x,section_id_x,price_x,title_y,section_id_y,price_y
0,60824,2021-02-12,2,25.0,50.0,1,0.693147,3.218876,NaN,NaN,NaN,NaN,NaN,NaN
1,60837,2021-02-12,1,175.0,175.0,1,0.000000,5.164786,NaN,NaN,NaN,NaN,NaN,NaN
2,60838,2021-02-12,1,160.0,160.0,1,0.000000,5.075174,NaN,NaN,NaN,NaN,NaN,NaN
3,60839,2021-02-12,1,160.0,160.0,1,0.000000,5.075174,NaN,NaN,NaN,NaN,NaN,NaN
4,60840,2021-02-12,1,160.0,160.0,1,0.000000,5.075174,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
panel_item_day.columns.tolist()


['id',
 'date',
 'qty',
 'avg_price',
 'revenue',
 'n_orders',
 'log_qty',
 'log_price',
 'title_x',
 'section_id_x',
 'price_x',
 'title_y',
 'section_id_y',
 'price_y']

In [21]:
panel_item_day.isna().sum()


id                    0
date                  0
qty                   0
avg_price             4
revenue               0
n_orders              0
log_qty               0
log_price             4
title_x         1999341
section_id_x    1999341
price_x         1999341
title_y         1999341
section_id_y    1999341
price_y         1999341
dtype: int64

In [22]:
menu_master.columns.tolist()


['id',
 'user_id',
 'created',
 'updated',
 'accounting_reference',
 'barcode',
 'deleted',
 'delivery',
 'demo_mode',
 'description',
 'discountable',
 'display_for_customers',
 'eat_in',
 'external_id',
 'image',
 'index',
 'number',
 'price',
 'purchases',
 'removable_ingredients',
 'section_id',
 'status',
 'takeaway',
 'title',
 'trainee_mode',
 'type',
 'variable_price',
 'vat',
 'voucher',
 'add_on_category_ids',
 'printer_category_ids',
 'standard_voucher_validity',
 'manage_inventory',
 'all_you_can_eat',
 'all_you_can_eat_item_ids',
 'all_you_can_eat_validity',
 'section_id_menu',
 'rating',
 'votes',
 'purchases_menu',
 'price_menu',
 'status_menu',
 'type_menu',
 'title_menu',
 'index_menu',
 'item_id',
 'order_count',
 'places']

In [23]:
# make IDs consistent
panel_item_day["id"] = panel_item_day["id"].astype(str)
menu_master["id"] = menu_master["id"].astype(str)

# pick the right columns from menu_master and rename them cleanly
mm = menu_master.rename(columns={
    "title": "item_title",
    "section_id": "menu_section_id",
    "price": "menu_list_price"
})

panel_item_day = panel_item_day.merge(
    mm[["id", "item_title", "menu_section_id", "menu_list_price"]],
    on="id",
    how="left"
)


In [24]:
panel_item_day[["id", "item_title", "menu_section_id", "menu_list_price"]].head(10)


,id,item_title,menu_section_id,menu_list_price
0,60824,NaN,NaN,NaN
1,60837,NaN,NaN,NaN
2,60838,NaN,NaN,NaN
3,60839,NaN,NaN,NaN
4,60840,NaN,NaN,NaN
5,60849,NaN,NaN,NaN
6,60850,NaN,NaN,NaN
7,60851,NaN,NaN,NaN
8,60860,NaN,NaN,NaN
9,60861,NaN,NaN,NaN


In [25]:
[c for c in menu_master.columns if "title" in c.lower() or "section" in c.lower() or "price" in c.lower()]


['price',
 'section_id',
 'title',
 'variable_price',
 'section_id_menu',
 'price_menu',
 'title_menu']

In [26]:
panel_item_day.columns.tolist()


['id',
 'date',
 'qty',
 'avg_price',
 'revenue',
 'n_orders',
 'log_qty',
 'log_price',
 'title_x',
 'section_id_x',
 'price_x',
 'title_y',
 'section_id_y',
 'price_y',
 'item_title',
 'menu_section_id',
 'menu_list_price']

In [27]:
[c for c in menu_master.columns if "title" in c.lower() or "section" in c.lower() or "price" in c.lower()]


['price',
 'section_id',
 'title',
 'variable_price',
 'section_id_menu',
 'price_menu',
 'title_menu']

In [28]:
menu_master = (
    dim_items
      .merge(dim_menu_items[["id","section_id","rating","votes","purchases","price","status","type","title","index"]],
             on="id", how="left", suffixes=("","_menu"))
      .merge(most_agg, left_on="id", right_on="item_id", how="left")
)


NameError: name 'dim_menu_items' is not defined